# Pooled EDA, no arm splits

The design doc has to be written and committed before any outcome is compared across arms, that commit is the pre-registration record. So everything below pools all three arms together. It never looks at `segment` against `visit`, `conversion`, or `spend`. The goal here is to check real numbers in the data (baseline rates, category values, distributions) that earlier design conversations assumed from memory, before locking them in.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from load import load_hillstrom

df = load_hillstrom()
print(f"rows: {len(df)}")
df.head()

rows: 64000


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,visit,conversion,spend,segment
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,0,0,0.0,Womens E-Mail
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,0,0,0.0,No E-Mail
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,0,0,0.0,Womens E-Mail
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,0,0,0.0,Mens E-Mail
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,0,0,0.0,Womens E-Mail


### Overall conversion rate

Pooled across all three arms. This is the number the MDE decision (0.5 percentage points absolute) was anchored to, assumed at roughly 0.9% from memory. Checking it here.

In [2]:
overall_conversion_rate = df["conversion"].mean()
overall_visit_rate = df["visit"].mean()
overall_spend_mean = df["spend"].mean()

print(f"overall conversion rate: {overall_conversion_rate:.4%}")
print(f"overall visit rate: {overall_visit_rate:.4%}")
print(f"overall mean spend: ${overall_spend_mean:.4f}")

overall conversion rate: 0.9031%
overall visit rate: 14.6781%
overall mean spend: $1.0509


### Segment category values

Checking the actual category values for `channel` and `zip_code`, assumed earlier as Phone/Web/Multichannel and Rural/Suburban/Urban. These feed the pre-registered segment list.

In [3]:
print("channel value counts:")
print(df["channel"].value_counts())
print()
print("zip_code value counts:")
print(df["zip_code"].value_counts())
print()
print("newbie value counts:")
print(df["newbie"].value_counts())

channel value counts:
channel
Web             28217
Phone           28021
Multichannel     7762
Name: count, dtype: int64

zip_code value counts:
zip_code
Surburban    28776
Urban        25661
Rural         9563
Name: count, dtype: int64

newbie value counts:
newbie
1    32144
0    31856
Name: count, dtype: int64


### Recency distribution

This feeds the recency segment cut decision, tertiles vs a fixed cutoff like <=3 / 4-6 / 7+ months. Need the real shape of this column to pick sensibly.

In [4]:
print(df["recency"].describe())
print()
print("value counts:")
print(df["recency"].value_counts().sort_index())
print()
tertile_edges = df["recency"].quantile([1 / 3, 2 / 3]).values
print(f"tertile edges (months): {tertile_edges}")

count    64000.000000
mean         5.763734
std          3.507592
min          1.000000
25%          2.000000
50%          6.000000
75%          9.000000
max         12.000000
Name: recency, dtype: float64

value counts:
recency
1     8952
2     7537
3     5904
4     5077
5     4510
6     4605
7     4078
8     3495
9     6441
10    7565
11    3504
12    2332
Name: count, dtype: int64

tertile edges (months): [3. 8.]


### Comparing recency bin schemes

Two candidate ways to cut recency into three segments: data-driven tertiles (<=3 / 4-8 / 9+, from the quantiles above) vs a fixed cutoff chosen before seeing the data (<=3 / 4-6 / 7+). Checking the resulting group sizes under each, since lopsided segment sizes mean uneven power across the segment comparisons.

In [5]:
def bin_sizes(edges, labels):
    bins = pd.cut(df["recency"], bins=edges, labels=labels, include_lowest=True)
    return bins.value_counts().sort_index()

tertile_sizes = bin_sizes([0, 3, 8, 12], ["<=3", "4-8", "9+"])
fixed_sizes = bin_sizes([0, 3, 6, 12], ["<=3", "4-6", "7+"])

print("tertile scheme (<=3 / 4-8 / 9+) group sizes:")
print(tertile_sizes)
print(f"largest / smallest ratio: {tertile_sizes.max() / tertile_sizes.min():.2f}")
print()
print("fixed cutoff scheme (<=3 / 4-6 / 7+) group sizes:")
print(fixed_sizes)
print(f"largest / smallest ratio: {fixed_sizes.max() / fixed_sizes.min():.2f}")

tertile scheme (<=3 / 4-8 / 9+) group sizes:
recency
<=3    22393
4-8    21765
9+     19842
Name: count, dtype: int64
largest / smallest ratio: 1.13

fixed cutoff scheme (<=3 / 4-6 / 7+) group sizes:
recency
<=3    22393
4-6    14192
7+     27415
Name: count, dtype: int64
largest / smallest ratio: 1.93


### CUPED covariate candidates

CUPED needs a pre-treatment covariate correlated with the outcome, measured before the email went out and unaffected by it. Candidates in this data: `history` (past 12-month spend), `recency` (months since last purchase), `mens`/`womens` (past purchase category flags), `newbie`. Checking pooled correlation with `conversion` (the primary metric) and `spend` (the secondary dollar metric) to pick the strongest one, rather than assuming.

In [6]:
candidates = ["history", "recency", "mens", "womens", "newbie"]

corr_table = pd.DataFrame({
    "corr_with_conversion": [df[c].corr(df["conversion"]) for c in candidates],
    "corr_with_spend": [df[c].corr(df["spend"]) for c in candidates],
}, index=candidates)

corr_table

,corr_with_conversion,corr_with_spend
history,0.029405,0.021729
recency,-0.024412,-0.016348
mens,0.002492,0.008599
womens,0.012702,0.002173
newbie,-0.011331,-0.007623


### Bootstrap resample count for spend

Spend is heavy-tailed (most customers spend $0, a few spend a lot), so the bootstrap CI on mean spend needs enough resamples for the percentile bounds to stabilize. Rather than assume a round number, run the bootstrap once with a fixed seed and check the CI bounds at increasing checkpoints, to see where they stop moving. Pooled data only, this checks the stability of a statistical procedure, it does not compare arms.

In [7]:
rng = np.random.default_rng(seed=42)
spend = df["spend"].to_numpy()
n = len(spend)

max_b = 20000
boot_means = np.empty(max_b)
for b in range(max_b):
    idx = rng.integers(0, n, size=n)
    boot_means[b] = spend[idx].mean()

checkpoints = [100, 500, 1000, 2000, 5000, 10000, 20000]
rows = []
for b in checkpoints:
    lo, hi = np.percentile(boot_means[:b], [2.5, 97.5])
    rows.append({"B": b, "ci_low": lo, "ci_high": hi, "ci_width": hi - lo})

convergence = pd.DataFrame(rows)
convergence["pct_change_in_width_vs_prev"] = convergence["ci_width"].pct_change() * 100
convergence

,B,ci_low,ci_high,ci_width,pct_change_in_width_vs_prev
0,100,0.947702,1.147675,0.199973,NaN
1,500,0.925591,1.153427,0.227836,13.933000
2,1000,0.926385,1.157219,0.230833,1.315679
3,2000,0.930414,1.162114,0.231700,0.375552
4,5000,0.934561,1.167864,0.233304,0.692082
5,10000,0.936718,1.168726,0.232008,-0.555189
6,20000,0.936173,1.168639,0.232466,0.197014
